# 🧩 TD10/TD12 — Application Orchestration & Market Snapshot (CLI)
## MarketPulse — Feature 4: `feature/market-snapshot-cli` ➔ `src/main.py`

In this notebook, we assemble the application's central orchestrator.

---
### 👥 Team Organization (4-Student Group)
* **Assigned Developer (PR Author)**: **Student 2** (Lead Orchestration)
* **Official Code Reviewer**: **Student 1**
* **Dedicated Git Branch**: `feature/market-snapshot-cli`
* **Assessment Milestone**: Central pivot of **Checkpoint C (45%)**
---

### 🎯 What is the Application Snapshot Contract?
The fundamental architectural principle of MarketPulse is the **Application Snapshot** (`docs/12_MARKETPULSE_APPLICATION_SNAPSHOT_CONTRACT.md`).
It is a standardized Python dictionary computed exactly once by the core orchestrator:
1. Loads instrument and benchmark metadata (`instruments.json`).
2. Acquires market observations via the selected provider (`fetch_bloomberg_prices`).
3. Aligns time series on common dates (`align_series`).
4. Computes normalized financial metrics (`analytics.py`).
5. **Exposes the exact same result**:
   - To the console terminal (`print_market_snapshot`).
   - To the interactive web dashboard (`dashboard.py`).

### 🎯 Notebook Objectives:
1. Implement `build_market_snapshot(provider_mode)`.
2. Implement formatted terminal output `print_market_snapshot(snapshot)`.
3. Test end-to-end execution.
4. Follow the **Git Integration Guide** to finalize **`src/main.py`**.


## 1. Imports and Metadata Resolution


In [ ]:
from pathlib import Path
import json
import sys

# Add search paths for brother modules
sys.path.insert(0, str(Path("..").resolve()))
sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, str(Path("src").resolve()))

# Robust resolution of instruments.json
CANDIDATE_DATA_DIRS = [
    Path("data/sample"),
    Path("../data/sample"),
    Path("../../data/sample"),
]


def get_instruments_file():
    for d in CANDIDATE_DATA_DIRS:
        candidate = d / "instruments.json"
        if candidate.exists():
            return candidate
    return CANDIDATE_DATA_DIRS[0] / "instruments.json"


LOOKBACK_LABEL = "1 month"
INTERVAL_LABEL = "Daily"
PROVIDER_LABEL = "Bloomberg stage - APPROVED_SAMPLE"


def load_instruments():
    """Load instrument and benchmark metadata from JSON."""
    with open(get_instruments_file(), encoding="utf-8") as file:
        return json.load(file)


## 2. Snapshot Constructor (`build_market_snapshot`)


In [ ]:
# Import analytics and provider functions
try:
    from src.analytics import (
        align_series,
        calculate_base_100,
        calculate_period_return,
        calculate_relative_performance,
    )
    from src.providers.bloomberg_provider import fetch_bloomberg_prices
except ImportError:
    print("ℹ️ Note: src/ modules not found directly. Ensure notebooks 01 and 03 have been executed.")


def build_market_snapshot(provider_mode="APPROVED_SAMPLE"):
    """
    Construct the unified market snapshot for MarketPulse.
    This application result is consumed identically by the terminal CLI
    and the interactive Dash dashboard.
    """
    instruments = load_instruments()
    instrument = instruments["instrument"]
    benchmark = instruments["benchmark"]

    # Acquisition via Bloomberg provider (APPROVED_SAMPLE by default)
    raw_instrument_prices = fetch_bloomberg_prices(instrument["ticker"], mode=provider_mode)
    raw_benchmark_prices = fetch_bloomberg_prices(benchmark["ticker"], mode=provider_mode)

    # Align series on common dates
    aligned_instrument, aligned_benchmark = align_series(
        raw_instrument_prices,
        raw_benchmark_prices,
    )

    # Compute quantitative metrics via analytics.py
    instrument_return = calculate_period_return(aligned_instrument)
    benchmark_return = calculate_period_return(aligned_benchmark)
    relative_perf = calculate_relative_performance(instrument_return, benchmark_return)
    instrument_base_100 = calculate_base_100(aligned_instrument)
    benchmark_base_100 = calculate_base_100(aligned_benchmark)

    provider_name = PROVIDER_LABEL if provider_mode == "APPROVED_SAMPLE" else "Bloomberg - LIVE"

    return {
        "provider": provider_name,
        "lookback": LOOKBACK_LABEL,
        "interval": INTERVAL_LABEL,
        "instrument": {
            "ticker": instrument["ticker"],
            "name": instrument["name"],
            "currency": instrument.get("currency", "USD"),
        },
        "benchmark": {
            "ticker": benchmark["ticker"],
            "name": benchmark["name"],
        },
        "instrument_return": instrument_return,
        "benchmark_return": benchmark_return,
        "relative_performance": relative_perf,
        "instrument_base_100": instrument_base_100,
        "benchmark_base_100": benchmark_base_100,
    }


## 3. Formatted Terminal Renderer (`print_market_snapshot`)


In [ ]:
def print_market_snapshot(snapshot):
    """Render market snapshot as clean, structured console output."""
    inst = snapshot["instrument"]
    bm = snapshot["benchmark"]

    print("=== MarketPulse ===")
    print()
    print("Market configuration")
    print(f"Provider : {snapshot['provider']}")
    print(f"Period   : {snapshot['lookback']}")
    print(f"Interval : {snapshot['interval']}")
    print()
    print("Instrument")
    print(f"{inst['ticker']} - {inst['name']}")
    print(f"Observations : {len(snapshot['instrument_base_100'])}")
    print(f"Period return: {snapshot['instrument_return']:+.2f}%")
    print()
    print("Benchmark")
    print(f"{bm['ticker']} - {bm['name']}")
    print(f"Observations : {len(snapshot['benchmark_base_100'])}")
    print(f"Period return: {snapshot['benchmark_return']:+.2f}%")
    print()
    print("Performance comparison")
    print(f"Relative performance: {snapshot['relative_performance']:+.2f} percentage points")


## 4. Test Execution of Terminal Snapshot


In [ ]:
# Generate snapshot and display
snapshot = build_market_snapshot()
print_market_snapshot(snapshot)


---
## 🛠️ Step-by-Step Git Integration Guide (Team of 4)

Follow these steps to integrate Feature 4:

### Step 1: Branch Preparation (by Student 2)
```bash
git switch main
git pull origin main
git switch -c feature/market-snapshot-cli
```

### Step 2: Upgrade `src/main.py`
1. Open **`src/main.py`**.
2. Replace the initial starter with this modular architecture:
   - Imports from `analytics` and `providers.bloomberg_provider`
   - Implementation of `build_market_snapshot()`
   - Implementation of `print_market_snapshot()`
   - Standard execution entry point:
     ```python
     def main():
         snapshot = build_market_snapshot()
         print_market_snapshot(snapshot)

     if __name__ == "__main__":
         main()
     ```

### Step 3: Local CLI Verification
```bash
python3 src/main.py
```
Verify the clean console output matching the official contract.

### Step 4: Commit, Push and Open PR
```bash
git add src/main.py
git commit -m "feat(orchestration): implement build_market_snapshot and cli summary"
git push -u origin feature/market-snapshot-cli
```
- Open the PR on GitHub (`feature/market-snapshot-cli` ➔ `main`).
- **Reviewer**: Student 1.
- 📸 **Checkpoint C Evidence (Student 2)**: Capture `03_final_pull_request.png` and `04_reproducible_run.png`.
- Student 1 reviews the snapshot contract adherence, approves and merges.
